In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pass del transpiler potenziati dall'AI

I pass del transpiler potenziati dall'AI sono pass che funzionano come sostituti diretti dei pass "tradizionali" di Qiskit per alcune operazioni di transpilazione. Spesso producono risultati migliori rispetto agli algoritmi euristici esistenti (ad esempio, profondità e conteggio CNOT inferiori), ma sono anche molto più veloci rispetto ad algoritmi di ottimizzazione come i risolutori di soddisfacibilità booleana. I pass del transpiler AI vengono eseguiti nel tuo ambiente locale.

> **Note:** I pass del transpiler potenziati dall'AI sono in stato di rilascio beta e soggetti a modifiche.
>     Se hai feedback o vuoi contattare il team di sviluppo, utilizza questo [canale dello Spazio Slack di Qiskit](https://qiskit.slack.com/archives/C06KF8YHUAU).

I seguenti pass sono attualmente disponibili:

**Pass di routing**
 - `AIRouting`: Selezione del layout e routing del circuito

**Pass di sintesi del circuito**
 - `AICliffordSynthesis`: Sintesi di circuiti Clifford
 - `AILinearFunctionSynthesis`: Sintesi di circuiti a funzione lineare
 - `AIPermutationSynthesis`: Sintesi di circuiti di permutazione

Per usare i pass del transpiler AI, prima installa il pacchetto `qiskit-ibm-transpiler`. Visita la [documentazione API di qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) per maggiori informazioni sulle diverse opzioni disponibili.

## Eseguire i pass del transpiler AI in locale o nel cloud
Prima installa `qiskit-ibm-transpiler` con alcune dipendenze aggiuntive come segue:

In [ ]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Dopo aver installato le dipendenze aggiuntive, la modalità predefinita per eseguire i pass del transpiler potenziati dall'AI è utilizzare il tuo computer locale.

## Pass di routing AI
Il pass `AIRouting` agisce sia come fase di layout che come fase di routing. Può essere utilizzato all'interno di un `PassManager` come segue:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [3]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

In questo esempio, il `backend` determina per quale coupling map effettuare il routing, il parametro `optimization_level` (1, 2 o 3) determina lo sforzo computazionale da impiegare nel processo (valori più alti di solito producono risultati migliori ma richiedono più tempo), e il parametro `layout_mode` specifica come gestire la selezione del layout.
Il parametro `layout_mode` include le seguenti opzioni:

- `keep`: Questa modalità rispetta il layout impostato dai pass del transpiler precedenti (o usa il layout triviale se non impostato). Viene tipicamente usata solo quando il circuito deve essere eseguito su qubit specifici del dispositivo. Spesso produce risultati peggiori perché ha meno margine per l'ottimizzazione.
- `improve`: Questa modalità utilizza il layout impostato dai pass del transpiler precedenti come punto di partenza. È utile quando hai una buona ipotesi iniziale per il layout; ad esempio, per circuiti costruiti in modo da seguire approssimativamente la coupling map del dispositivo. È utile anche se vuoi provare altri pass di layout specifici in combinazione con il pass `AIRouting`.
- `optimize`: Questa è la modalità predefinita. Funziona meglio per circuiti generali in cui potresti non avere buone ipotesi di layout. Questa modalità ignora le selezioni di layout precedenti.
- `local_mode`: Questo flag determina dove viene eseguito il pass `AIRouting`. Se `False`, `AIRouting` viene eseguito in remoto tramite il Qiskit Transpiler Service. Se `True`, il pacchetto tenta di eseguire il pass nel tuo ambiente locale, con un fallback alla modalità cloud se le dipendenze richieste non vengono trovate.

## Pass di sintesi del circuito AI
I pass di sintesi del circuito AI ti consentono di ottimizzare porzioni di diversi tipi di circuito ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) ri-sintetizzandole. Un modo tipico per utilizzare il pass di sintesi è il seguente: